In [204]:
import pandas as pd

In [205]:
tama = pd.read_pickle("../sqanti/tama_augustus_sqanti_full.pkl")

In [206]:
pasa = pd.read_pickle("../pasa/pasa_augustus_sqanti_full.pkl")

In [207]:
# prepare ref gene sets:
busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/busco_at9900.txt", header=None)
house = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/housekeeping_at9900.txt", header=None)
nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/nlrs_at9900.txt", header=None)

busco["type"] = "busco"
house["type"] = "housekeeping"
nlr["type"] = "nlr"
gene_type = pd.concat([busco,house,nlr])
gene_type.columns = ["ref_isoform","type"]
gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"

In [208]:
gene_type.head()

,ref_isoform,type,ref_gene
0,g10000.t1,busco,g10000
1,g10003.t1,housekeeping,g10003
2,g10010.t1,busco,g10010
3,g10011.t1,busco,g10011
4,g1002.t1,busco,g1002


In [209]:
tama_type = pd.merge(left=tama, right=gene_type, how='left', left_on='associated_transcript', right_on='ref_isoform')

## at9900 Tama

### general

In [210]:
t9900 = tama_type.loc[tama_type["accession"] == "at9900"].copy()
# novel transcripts
novel_transcripts = t9900["associated_transcript"].value_counts()["novel"]
# ref_genes wit novel transcripts
ref_genes_w_novel_transcripts=t9900.loc[t9900["associated_transcript"]=="novel", "associated_gene"].value_counts().index.unique()
len(ref_genes_w_novel_transcripts)
# tama_genes wit novel transcripts
tama_genes_w_novel_transcripts=t9900.loc[t9900["associated_transcript"]=="novel", "gene"].value_counts().index.unique()
len(tama_genes_w_novel_transcripts)
# novel genes
t9900["gene_nov"]= t9900.associated_gene.str.split("_").str[0]
novel_genes=t9900.loc[t9900["gene_nov"]=="novelGene"]["gene"].value_counts().index.unique().tolist()
len(novel_genes)
t9900.loc[t9900["associated_transcript"]=="novel", "subcategory"].value_counts()
t9900.loc[t9900["associated_transcript"]!="novel", "subcategory"].value_counts()

alternative_3end5end    62953
alternative_3end        21998
3prime_fragment         18871
mono-exon               15076
5prime_fragment          2036
intron_retention         2031
alternative_5end          942
reference_match           422
internal_fragment         294
Name: subcategory, dtype: int64

In [211]:
### subset:
for i in ["all", "nlr", "busco", "housekeeping"]:
    print(f"####{i}####")
    if i == "all":
        df = t9900
    else:
        df = t9900[t9900["type"] == i].copy()
    
    # Tama genes
    tama_genes = df["gene"].value_counts().index.unique()
    print(f"Tama genes: {len(tama_genes)}")
    
    # augustus genes
    ref_genes = df["associated_gene"].value_counts().index.unique()
    print(f"Ref genes: {len(ref_genes)}")
    
    # transcripts
    print(f"Transcripts: {len(df)}")
    
    # novel transcripts
    novel_transcripts = len(df.loc[df["associated_transcript"]=="novel"])
    print(f"Novel transcripts: {novel_transcripts}")
    
    # ref_genes wit novel transcripts
    ref_genes_w_novel_transcripts=df.loc[df["associated_transcript"]=="novel", "associated_gene"].value_counts().index.unique()
    len(ref_genes_w_novel_transcripts)
    print(f"Reference genes with novel transcripts: {len(ref_genes_w_novel_transcripts)}")
    
    # tama_genes wit novel transcripts
    tama_genes_w_novel_transcripts=df.loc[df["associated_transcript"]=="novel", "gene"].value_counts().index.unique()
    len(tama_genes_w_novel_transcripts)
    print(f"Tama genes with novel transcripts: {len(tama_genes_w_novel_transcripts)}")
    
    # novel genes
    df["gene_nov"]= df.associated_gene.str.split("_").str[0]
    novel_genes=df.loc[df["gene_nov"]=="novelGene"]["gene"].value_counts().index.unique().tolist()
    len(novel_genes)
    print(f"Novel genes: {len(novel_genes)}")
    
    print("Novel transcripts subategories:")
    print(df.loc[df["associated_transcript"]=="novel", "subcategory"].value_counts())
    
    print("Non novel transcripts subategories:")
    print(df.loc[df["associated_transcript"]!="novel", "subcategory"].value_counts())
    print("\n")

####all####
Tama genes: 16754
Ref genes: 18215
Transcripts: 197120
Novel transcripts: 72497
Reference genes with novel transcripts: 12269
Tama genes with novel transcripts: 11103
Novel genes: 846
Novel transcripts subategories:
at_least_one_novel_splicesite       41507
intron_retention                    14859
mono-exon                            7299
multi-exon                           6122
combination_of_known_splicesites     2660
mono-exon_by_intron_retention          50
Name: subcategory, dtype: int64
Non novel transcripts subategories:
alternative_3end5end    62953
alternative_3end        21998
3prime_fragment         18871
mono-exon               15076
5prime_fragment          2036
intron_retention         2031
alternative_5end          942
reference_match           422
internal_fragment         294
Name: subcategory, dtype: int64


####nlr####
Tama genes: 104
Ref genes: 104
Transcripts: 721
Novel transcripts: 0
Reference genes with novel transcripts: 0
Tama genes with novel tra

### at9900 Pasa

In [212]:
pasa

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,structural_category,subcategory,Sqanti_ORF_seq
0,g1,g1.t1,at6923,g1,g1.t1,6,1663,full-splice_match,alternative_3end5end,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYD...
1,g10,g10.t1,at6923,g10,novel,4,603,novel_not_in_catalog,at_least_one_novel_splicesite,MSTVGELACSYAVMILEDEGIAITADKIATLVKAAGVSIESYWPML...
2,g100,g100.t1,at6923,g100,g100.t1,1,1787,full-splice_match,mono-exon,MGSLVKAYYCNQMHHNSIIPHFHHYRPSSLQNPISLFAITSPSSSE...
3,g1000,g1000.t1,at6923,g1000,novel,5,1653,novel_not_in_catalog,at_least_one_novel_splicesite,MLSGNNGNTAPPVFMNENQLQYQTNLRSNQLHLLGNMGGGCTVDPV...
4,g1001,g1001.t1,at6923,g1001,novel,4,788,novel_not_in_catalog,at_least_one_novel_splicesite,NaN
...,...,...,...,...,...,...,...,...,...,...
45167,ATMG01350,ATMG01350.1,col0,ATMG01350,ATMG01350.1,1,438,full-splice_match,mono-exon,MTKREYNSQPEMKEEVLAYLLQLSASLVLPVAIWLIAAGQIFTCLR...
45168,ATMG01360,ATMG01360.1,col0,ATMG01360,ATMG01360.1,1,1584,full-splice_match,mono-exon,MKNLVRWLFSTNHKDIGTLYFIFGAIAGVMGTCFSVLIRMELARPG...
45169,ATMG01370,ATMG01370.1,col0,ATMG01370,ATMG01370.1,1,336,full-splice_match,mono-exon,NaN
45170,ATMG01400,ATMG01400.1,col0,ATMG01400,ATMG01400.1,1,318,full-splice_match,mono-exon,MKYHFSSMEPWWKREFSFCIPAIYIKMASISLFQNSWLKMKHLPSC...


In [213]:
pasa_type = pd.merge(left=pasa, right=gene_type, how='left', left_on='associated_transcript', right_on='ref_isoform')

In [214]:
p9900 = pasa_type.loc[pasa_type["accession"] == "at9900"].copy()

In [215]:
p9900

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,structural_category,subcategory,Sqanti_ORF_seq,ref_isoform,type,ref_gene
762768,g1,g1.t1,at9900,g1,g1.t1,6,1685,full-splice_match,alternative_3end5end,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYD...,NaN,NaN,NaN
762769,g10,g10.t1,at9900,g10,novel,4,612,novel_not_in_catalog,at_least_one_novel_splicesite,MSTVGELACSYAVMILEDEGIAITADKIATLVKAAGVSIESYWPML...,NaN,NaN,NaN
762770,g100,g100.t1,at9900,g100,g100.t1,1,1626,full-splice_match,mono-exon,MKLACFVFLILSSLVSSSLATAPPNASIYESFLQCFSNQTGAPPEK...,NaN,NaN,NaN
762771,g1000,g1000.t1,at9900,g1000,g1000.t1,1,2398,full-splice_match,mono-exon,MIHTKTGSGRRILTFPTVEPSESISIVTLLDSLIQLAGDILTFKSK...,NaN,NaN,NaN
762772,g1001,g1001.t1,at9900,g1001,novel,14,1942,novel_not_in_catalog,at_least_one_novel_splicesite,MKRQRAIELDRVKKTMLNIDWDDALGDEEVPELEIIATDKIPPREP...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
810275,g31852,g31852.t1,at9900,g31852,g31852.t1,2,1203,full-splice_match,reference_match,MSTPAQSDLVKVFTDVASNYRGTSDIDLHAVYRDMRANSPVLQDNF...,NaN,NaN,NaN
810276,g31853,g31853.t1,at9900,g31853,g31853.t1,1,1308,full-splice_match,mono-exon,MKVEDLILVSVDDHAIEPPNAFARHMPARFKGREPHVIKKGDRDVW...,NaN,NaN,NaN
810277,g31854,g31854.t1,at9900,g31854,g31854.t1,1,1230,full-splice_match,mono-exon,MAAVERLPFCGRFVRSDCACVERERAHMAISQFSYSSIPPEAEALR...,NaN,NaN,NaN
810278,g31855,g31855.t1,at9900,g31855,g31855.t1,2,1767,full-splice_match,reference_match,MGCEGGEWPAELWAALDESGFTQAGVTEALGGAEASWADMYVLVRA...,g31855.t1,busco,g31855


In [221]:
### subset:
for i in ["all", "nlr", "busco", "housekeeping"]:
    print(f"####{i}####")
    if i == "all":
        df_a = p9900
    else:
        df_a = p9900[p9900["type"] == i].copy()
    
    df = df_a.loc[df_a["subcategory"] != "reference_match"].copy()
    #df = df_a
    
    # Pasa genes
    pasa_genes = df["gene"].value_counts().index.unique()
    print(f"Pasa genes: {len(pasa_genes)}")
    
    # augustus genes
    ref_genes = df["associated_gene"].value_counts().index.unique()
    print(f"Ref genes: {len(ref_genes)}")
    
    # transcripts
    print(f"Transcripts: {len(df)}")
    
    # novel transcripts
    novel_transcripts = len(df.loc[df["associated_transcript"]=="novel"])
    print(f"Novel transcripts: {novel_transcripts}")
    
    # ref_genes wit novel transcripts
    ref_genes_w_novel_transcripts=df.loc[df["associated_transcript"]=="novel", "associated_gene"].value_counts().index.unique()
    len(ref_genes_w_novel_transcripts)
    print(f"Reference genes with novel transcripts: {len(ref_genes_w_novel_transcripts)}")
    
    # pasa_genes wit novel transcripts
    pasa_genes_w_novel_transcripts=df.loc[df["associated_transcript"]=="novel", "gene"].value_counts().index.unique()
    len(pasa_genes_w_novel_transcripts)
    print(f"Pasa genes with novel transcripts: {len(pasa_genes_w_novel_transcripts)}")
    
    # novel genes
    df["gene_nov"]= df.associated_gene.str.split("_").str[0]
    novel_genes=df.loc[df["gene_nov"]=="novelGene"]["gene"].value_counts().index.unique().tolist()
    len(novel_genes)
    print(f"Novel genes: {len(novel_genes)}")
    
    print("Novel transcripts subategories:")
    print(df.loc[df["associated_transcript"]=="novel", "subcategory"].value_counts())
    
    print("Non novel transcripts subategories:")
    print(df.loc[df["associated_transcript"]!="novel", "subcategory"].value_counts())
    print("\n")

####all####
Pasa genes: 20857
Ref genes: 20858
Transcripts: 36551
Novel transcripts: 17645
Reference genes with novel transcripts: 8104
Pasa genes with novel transcripts: 8103
Novel genes: 0
Novel transcripts subategories:
intron_retention                    8371
at_least_one_novel_splicesite       8015
multi-exon                           851
combination_of_known_splicesites     408
Name: subcategory, dtype: int64
Non novel transcripts subategories:
alternative_3end5end    8057
mono-exon               7611
alternative_3end        1457
intron_retention        1313
3prime_fragment          243
5prime_fragment          189
alternative_5end          34
internal_fragment          2
Name: subcategory, dtype: int64


####nlr####
Pasa genes: 93
Ref genes: 93
Transcripts: 102
Novel transcripts: 0
Reference genes with novel transcripts: 0
Pasa genes with novel transcripts: 0
Novel genes: 0
Novel transcripts subategories:
Series([], Name: subcategory, dtype: int64)
Non novel transcripts subatego